In [1]:
import os
import boto3
from sagemaker import get_execution_role
import shutil
from pprint import pprint
import time
import pandas as pd

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'gen-xii-retro-scoring'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy functions.py
COPY functions.py ${LAMBDA_TASK_ROOT}

# copy api.py
COPY api.py ${LAMBDA_TASK_ROOT}

# copy parser
COPY cls_parser.pkl ${LAMBDA_TASK_ROOT}

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

tqdm==4.64.1
numpy==1.23.4
pandas==1.2.4
boto3==1.24.59

Writing requirements.txt


### Copy files

In [5]:
list_str_filename = [
    'functions.py',
    'api.py',
    'cls_parser.pkl',
]
for str_filename in list_str_filename:
    # logic
    if str_filename == 'cls_parser.pkl':
        str_source = f'../01_create_parser/output/{str_filename}'
    else:
        str_source = f'../01_create_parser/{str_filename}'
    str_destination = f'./{str_filename}'
    # copy
    shutil.copyfile(str_source, str_destination)

### Write ```lambda_function.py```

In [6]:
%%writefile lambda_function.py

import pandas as pd
import pickle
import json

# lambda handler
def lambda_handler(event, context):
    # get the input
    int_rows_to_parse = int(event['row'])
    print(f'Parsing rows: {int_rows_to_parse}')
    
    # constants
    str_project = '20231010-gen-xii'
        
    # load requests
    print('Loading requests...')
    str_filename = f'df_rows_{int_rows_to_parse}.gzip'
    str_uri = f's3://{str_project}/08_retro_scoring/03_step_function/rows/{str_filename}'
    df = pd.read_parquet(str_uri)
    print(f'There are {df.shape[0]} requests for this lambda function to parse')
    
    # load parser
    print('Loading parser...')
    str_filename = 'cls_parser.pkl'
    str_local_path = f'./{str_filename}'
    cls_parser = pickle.load(open(str_local_path, 'rb'))
    
    # parse (get income, LN, and TU)
    print('Parsing requests...')
    list_X_raw = []
    for a, str_request in enumerate(df['strRequest']):
        # get bigAccountId
        int_bigaccountid = df['bigAccountId'].iloc[a]
        # get dtmFunded
        dtm_funded = df['dtmFunded'].iloc[a]
    
        # convert string request to dict
        dict_json_request = json.loads(str_request)
        # get data
        cls_parser.get_data(dict_json_request)
        # parse data
        cls_parser.parse_data()
        # create X
        cls_parser.create_x()
        # get X_raw
        X_raw = cls_parser.dict_output['X_raw']
        
        # assign
        X_raw['bigAccountId'] = int_bigaccountid
        X_raw['dtmFunded'] = dtm_funded
        # get nrows
        int_nrows = X_raw.shape[0]
        if int_nrows == 1:
            X_raw['BITDEBTOR'] = 1
        else:
            X_raw['BITDEBTOR'] = [1,0]
        
        # append
        list_X_raw.append(X_raw)
    
    # concat
    print('Concatenating raw data...')
    X_raw = pd.concat(list_X_raw)
    
    # save memeory
    del list_X_raw
    
    # write to s3 as parquet
    print('Writing raw data to s3...')
    # set nonnumeric to string
    for col in X_raw.columns:
        # if not numeric
        if X_raw[col].dtype not in ['int64', 'float64']:
            # set as string
            X_raw[col] = X_raw[col].astype(str)
        else:
            pass
    # write to gzip
    str_filename = f'X_raw_{int_rows_to_parse}.gzip'
    str_uri = f's3://{str_project}/08_retro_scoring/03_step_function/parsed/{str_filename}'
    X_raw.to_parquet(str_uri, compression='gzip')

Writing lambda_function.py


### Build image and push to ECR

In [7]:
%%sh

# name the image
image=gen-xii-retro-scoring

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  94.21kB
Step 1/9 : FROM public.ecr.aws/lambda/python:3.8
3.8: Pulling from lambda/python
396455a9302a: Pulling fs layer
3b3352b76965: Pulling fs layer
9434e65b5620: Pulling fs layer
2a9f1ae020e1: Pulling fs layer
a759a1ee00b5: Pulling fs layer
f61cfb51b0a9: Pulling fs layer
2a9f1ae020e1: Waiting
a759a1ee00b5: Waiting
f61cfb51b0a9: Waiting
9434e65b5620: Verifying Checksum
9434e65b5620: Download complete
3b3352b76965: Verifying Checksum
3b3352b76965: Download complete
2a9f1ae020e1: Verifying Checksum
2a9f1ae020e1: Download complete
f61cfb51b0a9: Verifying Checksum
f61cfb51b0a9: Download complete
a759a1ee00b5: Verifying Checksum
a759a1ee00b5: Download complete
396455a9302a: Verifying Checksum
396455a9302a: Download complete
396455a9302a: Pull complete
3b3352b76965: Pull complete
9434e65b5620: Pull complete
2a9f1ae020e1: Pull complete
a759a1ee00b5: Pull complete
f61cfb51b0a9: Pull complete
Digest: sha256:7ee7718ee6b70099843bfc160ded9813e280f069c18ab4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.5/505.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.9/240.9 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.8/308.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 11.3 MB/s eta 0:00:00
Removing intermediate container 5de829041ac7
 ---> 5030808e3fd1
Step 5/9 : COPY functions.py ${LAMBDA_TASK_ROOT}
 ---> 744b6a1ac4a9
Step 6/9 : COPY api.py ${LAMBDA_TASK_ROOT}
 ---> ead2677929c2
Step 7/

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'gen-xii-retro-scoring' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/gen-xii-retro-scoring]
b934fa4463b9: Preparing
1293b3f6edce: Preparing
4a350562849b: Preparing
98a8c937ed31: Preparing
51f405875c37: Preparing
d7e2a6be8c41: Preparing
79525ec1a9b4: Preparing
ad8685f71a58: Preparing
9a585a1ad308: Preparing
7393ae547845: Preparing
09ccadc85d60: Preparing
5a148847bee7: Preparing
1d672e8b43a1: Preparing
9a585a1ad308: Waiting
7393ae547845: Waiting
1d672e8b43a1: Waiting
d7e2a6be8c41: Waiting
5a148847bee7: Waiting
79525ec1a9b4: Waiting
ad8685f71a58: Waiting
1293b3f6edce: Pushed
4a350562849b: Pushed
98a8c937ed31: Pushed
b934fa4463b9: Pushed
d7e2a6be8c41: Pushed
7393ae547845: Layer already exists
09ccadc85d60: Pushed
5a148847bee7: Pushed
79525ec1a9b4: Pushed
ad8685f71a58: Pushed
9a585a1ad308: Pushed
1d672e8b43a1: Pushed
51f405875c37: Pushed
latest: digest: sha256:f1d9654b2711e4da294d55508d79f2eaaa85106bc7ea70274621469c1dd3ea83 size: 3044


### 2. Create lambda function from image

In [8]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [9]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [10]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 07 Mar 2024 21:07:49 GMT',
                                      'x-amzn-requestid': '2c5ff968-c2be-4a63-bc32-7612e431044a'},
                      'HTTPStatusCode': 204,
                      'RequestId': '2c5ff968-c2be-4a63-bc32-7612e431044a',
                      'RetryAttempts': 0}}


In [11]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=900, # 15 minutes is maximum
    MemorySize=1000, # 1000 mb == 1 gb
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 1000, # 1000 mb == 1 gb
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': 'f1d9654b2711e4da294d55508d79f2eaaa85106bc7ea70274621469c1dd3ea83',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 1000},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:gen-xii-retro-scoring',
 'FunctionName': 'gen-xii-retro-scoring',
 'LastModified': '2024-03-07T21:07:49.187+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/gen-xii-retro-scoring'},
 'MemorySize': 1000,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1199',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 07 Mar 2024 21:07:50 GMT',
                                      'x-amzn-requestid': '6bf2939d-d291-4d2c-bebf-a2078254a0ba'},
                      'HTTPStatusCode': 201,
                      'RequestId': '6bf2939d-d291-4d2

### Clean-up

In [12]:
list_str_filename = list_str_filename + ['Dockerfile', 'lambda_function.py', 'requirements.txt']

for str_file in list_str_filename:
    os.remove(str_file)